# THS Ring Polymer — Analysis
Block averaging per Rg² e B₂/Rg³

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

BASE = Path('../runs')
RUNS = {
    100:  BASE / 'ring100'  / 'ring100.DAT',
    250:  BASE / 'ring250'  / 'ring250.DAT',
    500:  BASE / 'ring500'  / 'ring500.DAT',
    1000: BASE / 'ring1000' / 'ring1000.DAT',
}
# Columns: IB Rg2_1 Rg2_2 Ree2_1 Ree2_2 B2 acc_1 acc_2
data = {N: pd.read_csv(p, sep=r'\s+') for N, p in RUNS.items() if p.exists()}
print('Loaded:', {N: len(df) for N, df in data.items()})


## 1. Time series — scegli N_EQ

In [ ]:
# N_EQ come frazione dei blocchi disponibili (default 20%)
N_EQ_FRAC = 0.20   # oppure metti un numero fisso: N_EQ = 500

N_EQ = {N: max(1, int(len(df) * N_EQ_FRAC)) for N, df in data.items()}
print('N_EQ per run:', N_EQ)

fig, axes = plt.subplots(2, len(data), figsize=(4*len(data), 6), sharey='row')
if len(data) == 1:
    axes = axes.reshape(2, 1)

for col, (N, df) in enumerate(sorted(data.items())):
    neq = N_EQ[N]
    rg2 = (df['Rg2_1'] + df['Rg2_2']) / 2
    axes[0, col].plot(df['IB'], rg2, lw=0.5)
    axes[0, col].axvline(df['IB'].iloc[neq], color='r', ls='--', label=f'N_EQ={neq}')
    axes[0, col].set_title(f'N={N}  ({len(df)} blk)')
    axes[0, col].set_xlabel('block')
    if col == 0:
        axes[0, col].set_ylabel(r'$R_g^2$')
    axes[1, col].plot(df['IB'], df['B2'], lw=0.5, color='tomato')
    axes[1, col].axvline(df['IB'].iloc[neq], color='r', ls='--')
    axes[1, col].set_xlabel('block')
    if col == 0:
        axes[1, col].set_ylabel(r'$B_2$')

axes[0, 0].legend()
plt.tight_layout()
plt.savefig('timeseries.png', bbox_inches='tight')
plt.show()


## 2. Block analysis (Flyvbjerg–Petersen)
Divide i dati in blocchi di size L. Il **plateau** per L ≫ τ dà l'errore corretto.

In [ ]:
def block_error(x, min_blocks=8):
    x = np.asarray(x, dtype=float)
    n, Ls, errs, L = len(x), [], [], 1
    while n // L >= min_blocks:
        nb = n // L
        bm = x[:nb*L].reshape(nb, L).mean(axis=1)
        Ls.append(L)
        errs.append(bm.std(ddof=1) / np.sqrt(nb))
        L += max(1, L // 8)
    if not Ls:   # meno di min_blocks dati: ritorna stima naive
        return np.array([1]), np.array([x.std(ddof=1) / np.sqrt(max(len(x),1))])
    return np.array(Ls), np.array(errs)

def plateau_error(x, min_blocks=8):
    """Media globale + errore dal plateau (mediana top-25% block sizes)."""
    if len(x) == 0:
        return np.nan, np.nan
    Ls, errs = block_error(x, min_blocks)
    cut = max(1, int(0.75 * len(errs)))
    return np.mean(x), np.median(errs[cut:])


In [ ]:
# Blocking plot per Rg2 — cerca il plateau
fig, axes = plt.subplots(1, len(data), figsize=(4*len(data), 3.5))
if len(data) == 1:
    axes = [axes]

for ax, (N, df) in zip(axes, sorted(data.items())):
    neq = N_EQ[N]
    eq  = df.iloc[neq:]
    if len(eq) < 8:
        ax.set_title(f'N={N}: troppo pochi dati ({len(eq)} blk)')
        continue
    rg2 = ((eq['Rg2_1'] + eq['Rg2_2']) / 2).values
    Ls, errs = block_error(rg2)
    ax.semilogx(Ls, errs / errs[0], 'o-', ms=3, lw=0.8)
    ax.axhline(1, color='grey', ls=':')
    ax.set_title(f'N={N}  ({len(eq)} blk)')
    ax.set_xlabel('block size L')
    if ax is axes[0]:
        ax.set_ylabel(r'$\hat{\sigma}(L)/\hat{\sigma}(1)$')

plt.suptitle(r'Blocking per $R_g^2$ — plateau = errore corretto', y=1.02)
plt.tight_layout()
plt.savefig('blocking.png', bbox_inches='tight')
plt.show()


## 3. Risultati: ⟨Rg²⟩ e B₂/Rg³

**Rapporto delle medie globali**: B₂/Rg³ = ⟨B₂⟩ / ⟨Rg²⟩^(3/2)

**Errore** per propagazione:
σ(B₂/Rg³) = (B₂/Rg³) × √[(σ_B₂/B₂)² + (3/2 · σ_Rg²/Rg²)²]

**BLOCK_SIZE**: aumenta se il plateau parte a L > 1.

In [ ]:
BLOCK_SIZE = 1   # aumenta se vedi autocorrelazioni

def analyse(df, neq, bs=BLOCK_SIZE):
    eq  = df.iloc[neq:]
    if len(eq) < 8:
        print(f'  WARNING: solo {len(eq)} blocchi dopo termalizzazione')
    rg2 = ((eq['Rg2_1'] + eq['Rg2_2']) / 2).values
    b2  = eq['B2'].values
    if bs > 1:
        nb  = len(rg2) // bs
        rg2 = rg2[:nb*bs].reshape(nb, bs).mean(1)
        b2  = b2[:nb*bs].reshape(nb, bs).mean(1)

    rg2_m, rg2_e = plateau_error(rg2)
    b2_m,  b2_e  = plateau_error(b2)

    # Rapporto delle medie globali
    ratio   = b2_m / rg2_m**1.5
    ratio_e = ratio * np.sqrt((b2_e / b2_m)**2 + (1.5 * rg2_e / rg2_m)**2)

    return dict(rg2=rg2_m, rg2_e=rg2_e,
                b2=b2_m,   b2_e=b2_e,
                ratio=ratio, ratio_e=ratio_e,
                n=len(rg2))

res = {N: analyse(df, N_EQ[N]) for N, df in data.items()}

print(f"{'N':>6}  {'Rg2':>10} {'+-':>8}  {'B2':>10} {'+-':>8}  {'B2/Rg3':>8} {'+-':>7}  n")
print('-' * 75)
for N, r in sorted(res.items()):
    print(f"{N:6d}  {r['rg2']:10.4f} {r['rg2_e']:8.4f}  "
          f"{r['b2']:10.2f} {r['b2_e']:8.2f}  "
          f"{r['ratio']:8.4f} {r['ratio_e']:7.4f}  {r['n']}")


In [ ]:
if len(res) > 1:
    Ns   = np.array(sorted(res))
    rg2  = np.array([res[N]['rg2']     for N in Ns])
    rg2e = np.array([res[N]['rg2_e']   for N in Ns])
    rat  = np.array([res[N]['ratio']    for N in Ns])
    re   = np.array([res[N]['ratio_e'] for N in Ns])

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))

    a1.errorbar(Ns, rg2, rg2e, fmt='o-', capsize=4)
    a1.set_xscale('log'); a1.set_yscale('log')
    a1.set_xlabel('N'); a1.set_ylabel(r'$\langle R_g^2 \rangle$')
    if len(Ns) >= 3:
        sl, ic = np.polyfit(np.log(Ns), np.log(rg2), 1)
        a1.plot(Ns, np.exp(ic)*Ns**sl, '--k', label=f'slope={sl:.3f}')
        a1.legend()
    a1.set_title(r'$R_g^2$ vs $N$  (atteso $2\nu\approx1.176$)')

    a2.errorbar(Ns, rat, re, fmt='s-', color='tomato', capsize=4)
    a2.axhline(5.43, ls='--', color='grey', label='5.43 (SAW lineare, ref.)')
    a2.set_xscale('log')
    a2.set_xlabel('N'); a2.set_ylabel(r'$B_2/R_g^3$')
    a2.set_title(r'Ampiezza universale $B_2/R_g^3$')
    a2.legend()
    plt.tight_layout()
    plt.savefig('scaling.png', bbox_inches='tight')
    plt.show()
else:
    N, r = list(res.items())[0]
    print(f"N={N}: Rg2={r['rg2']:.4f}+-{r['rg2_e']:.4f}, "
          f"B2/Rg3={r['ratio']:.4f}+-{r['ratio_e']:.4f}")


## 4. Finite-size scaling: Rg²/N^(2ν) e B₂/Rg² vs 1/N

Con d/b = 0.433 le *leading* corrections-to-scaling si annullano (modello Domb-Joyce).
- **Rg²/N^(2ν)** → ampiezza universale A_ring per N→∞
- **B₂/Rg²** ~ N^ν, slope log-log = –ν (per vedere se siamo nel regime asintotico)

In [ ]:
if len(res) < 2:
    print("Servono almeno 2 run per il finite-size scaling.")
else:
    NU = 0.5876          # esponente 3D SAW (Nienhuis / Domb-Joyce)
    TWO_NU = 2 * NU

    Ns   = np.array(sorted(res))
    rg2  = np.array([res[N]['rg2']   for N in Ns])
    rg2e = np.array([res[N]['rg2_e'] for N in Ns])
    b2   = np.array([res[N]['b2']    for N in Ns])
    b2e  = np.array([res[N]['b2_e']  for N in Ns])

    inv_N = 1.0 / Ns

    # ── y1: Rg² / N^(2ν) ──────────────────────────────────────────────────
    y1  = rg2  / Ns**TWO_NU
    y1e = rg2e / Ns**TWO_NU

    # ── y2: B2 / Rg² ──────────────────────────────────────────────────────
    y2  = b2  / rg2
    y2e = y2 * np.sqrt((b2e/b2)**2 + (rg2e/rg2)**2)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.5))

    # ── Plot 1 ────────────────────────────────────────────────────────────
    a1.errorbar(inv_N, y1, y1e, fmt='o', capsize=4, zorder=3)
    if len(Ns) >= 2:
        sl1, ic1 = np.polyfit(np.log(inv_N), np.log(y1), 1)
        x_fit = np.array([inv_N.min()*0.8, inv_N.max()*1.2])
        a1.plot(x_fit, np.exp(ic1)*x_fit**sl1, '--k',
                label=f'slope = {sl1:.3f}')
        a1.legend(fontsize=10)
    a1.set_xscale('log'); a1.set_yscale('log')
    a1.set_xlabel(r'$1/N$', fontsize=12)
    a1.set_ylabel(r'$\langle R_g^2 \rangle / N^{2\nu}$', fontsize=12)
    a1.set_title(rf'$\nu = {NU}$ — dovrebbe $\to$ costante per $N\to\infty$',
                 fontsize=10)

    # tick labels con valori N
    a1.set_xticks(inv_N)
    a1.set_xticklabels([f'1/{N}' for N in Ns], fontsize=9)

    # ── Plot 2 ────────────────────────────────────────────────────────────
    a2.errorbar(inv_N, y2, y2e, fmt='s', color='tomato', capsize=4, zorder=3)
    if len(Ns) >= 2:
        sl2, ic2 = np.polyfit(np.log(inv_N), np.log(y2), 1)
        a2.plot(x_fit, np.exp(ic2)*x_fit**sl2, '--k',
                label=f'slope = {sl2:.3f}  (atteso −{NU:.3f})')
        a2.legend(fontsize=10)
    a2.set_xscale('log'); a2.set_yscale('log')
    a2.set_xlabel(r'$1/N$', fontsize=12)
    a2.set_ylabel(r'$B_2 / \langle R_g^2 \rangle$', fontsize=12)
    a2.set_title(r'$B_2/R_g^2 \sim N^\nu$ — slope log-log $= -\nu$', fontsize=10)

    a2.set_xticks(inv_N)
    a2.set_xticklabels([f'1/{N}' for N in Ns], fontsize=9)

    plt.tight_layout()
    plt.savefig('fss.png', bbox_inches='tight')
    plt.show()

    print(f"\nν usato = {NU}")
    print(f"Slope Rg²/N^(2ν) vs 1/N:  {sl1:.4f}  (ideale → 0 a d/b=0.433)")
    print(f"Slope B₂/Rg²    vs 1/N:  {sl2:.4f}  (atteso −{NU:.4f})")
